# EDA — `bronze.index_prices`

The benchmark side. SPY is what every trust is measured against; IVV and VOO exist only
as a sanity check that three trackers of the same index move together.

The important job of this notebook is the **contrast with the trusts**: `Adj_Close` works
correctly here and does not work there. That asymmetry is the whole reason Silver builds
total return by hand.

## 1. Shape and coverage

In [0]:
%sql
SELECT symbol,
       COUNT(*)                AS bars,
       SUBSTRING(MIN(`Date`), 1, 7) AS first_month,
       SUBSTRING(MAX(`Date`), 1, 7) AS last_month
FROM `index-vs-trust-pipeline`.bronze.index_prices
GROUP BY symbol
ORDER BY bars DESC;

**SPY 405 bars from 1993-01, IVV 317 from 2000-05, VOO 193 from 2010-09.** SPLG has no
rows at all — Yahoo does not serve it, and the pull log records the refusal.

SPY comfortably covers the 15-year horizon, which is all the benchmark has to do.

## 2. The same quality checks as the trusts

In [0]:
%sql
WITH base AS (
  SELECT symbol,
         SUBSTRING(`Date`, 1, 7) AS month_key,
         CAST(`Close` AS DOUBLE) AS close
  FROM `index-vs-trust-pipeline`.bronze.index_prices
),
with_median AS (
  SELECT *,
         PERCENTILE_APPROX(close, 0.5) OVER (
           PARTITION BY symbol ORDER BY month_key
           ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING
         ) AS local_median
  FROM base
)
SELECT SUM(CASE WHEN close IS NULL OR close <= 0 THEN 1 ELSE 0 END) AS bad_prices,
       SUM(CASE WHEN close / local_median > 2
                  OR close / local_median < 0.5 THEN 1 ELSE 0 END)  AS scale_outliers,
       COUNT(*)                                                     AS rows
FROM with_median
WHERE local_median > 0;

**Zero bad prices and zero scale outliers.** The corruption that affects 36 UK trusts
does not touch the US-listed trackers, so the benchmark side needs no repair.

That is worth saying out loud in the defence: the messy data is on one side only, and the
cleaning cannot therefore be accused of tilting the comparison.

## 3. `Adj_Close` — the contrast that matters

For the trusts, `Adj_Close` was within 5% of `Close` at the oldest bar for 97 of 100
symbols, meaning no dividends had been applied. Run the identical check here.

In [0]:
%sql
WITH first_bar AS (
  SELECT symbol,
         CAST(`Close` AS DOUBLE)   AS close,
         CAST(Adj_Close AS DOUBLE) AS adj_close,
         ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY `Date`) AS rn
  FROM `index-vs-trust-pipeline`.bronze.index_prices
)
SELECT symbol,
       ROUND(close, 2)             AS close_at_oldest_bar,
       ROUND(adj_close, 2)         AS adj_close_at_oldest_bar,
       ROUND(adj_close / close, 3) AS ratio
FROM first_bar
WHERE rn = 1
ORDER BY symbol;

In [0]:
%sql
-- SPY over ten years, measured both ways. The gap is the dividends.
WITH bounds AS (
  SELECT MIN(`Date`) AS start_date, MAX(`Date`) AS end_date
  FROM `index-vs-trust-pipeline`.bronze.index_prices
  WHERE symbol = 'SPY' AND `Date` >= DATE_SUB(CURRENT_DATE(), 3653)
),
endpoints AS (
  SELECT
    MAX(CASE WHEN p.`Date` = b.start_date THEN CAST(p.`Close` AS DOUBLE) END)   AS close_then,
    MAX(CASE WHEN p.`Date` = b.end_date   THEN CAST(p.`Close` AS DOUBLE) END)   AS close_now,
    MAX(CASE WHEN p.`Date` = b.start_date THEN CAST(p.Adj_Close AS DOUBLE) END) AS adj_then,
    MAX(CASE WHEN p.`Date` = b.end_date   THEN CAST(p.Adj_Close AS DOUBLE) END) AS adj_now
  FROM `index-vs-trust-pipeline`.bronze.index_prices p CROSS JOIN bounds b
  WHERE p.symbol = 'SPY'
)
SELECT ROUND(100 * (close_now / close_then - 1), 1) AS price_return_pct,
       ROUND(100 * (adj_now / adj_then - 1), 1)     AS total_return_pct,
       ROUND(100 * (adj_now / adj_then - close_now / close_then), 1) AS dividend_gap_pts
FROM endpoints;

**`Adj_Close` works properly for the index.** SPY's oldest bar shows a ratio well below
1, and over ten years the price return is about **+252%** against a total return of about
**+312%** — a gap of roughly **60 percentage points**, which is the dividends.

So the same column is trustworthy on one side of the comparison and useless on the other.
Using it where it works and not where it does not would measure the two sides by
different rules — which is precisely the bias this project exists to avoid. **Silver
builds total return the same way on both sides, from `Close` + `Dividends`.**

In [0]:
%sql
-- The dividends Silver will actually use for the benchmark.
SELECT COUNT(*)                                 AS dividend_months,
       ROUND(SUM(CAST(Dividends AS DOUBLE)), 2) AS total_paid_usd,
       SUBSTRING(MIN(`Date`), 1, 7)             AS first_dividend
FROM `index-vs-trust-pipeline`.bronze.index_prices
WHERE symbol = 'SPY' AND CAST(Dividends AS DOUBLE) > 0;

SPY has paid quarterly since 1993, so expect well over 100 dividend months. A zero here
would mean `actions=True` had been dropped from the pull and total return was impossible.

---

## Findings

| # | Finding | Status |
|---|---|---|
| 4.1 | SPY 405 bars from 1993-01; IVV 317; VOO 193; SPLG none. SPY covers every horizon. | **settled** |
| 4.2 | No bad prices and no scale outliers — the corruption is on the trust side only. | **settled** |
| 4.3 | `Adj_Close` **does** work for SPY: 10-year price return ~+252% vs total ~+312%, a ~60pt dividend gap. | **settled** |
| 4.4 | Because it works here and not for the trusts, it cannot be used at all — both sides must be measured the same way. | **settled** |
| 4.5 | SPY's dividend months are present and usable as the benchmark's income. | **settled** |
| 4.6 | The current month is partial here too, exactly as for the trusts. | **open** |

The only open item is shared with the trust table, so Silver handles it once for both.